# 00 — Overview

**microgpt on PYNQ-Z2: a char-level GPT baked entirely into PL fabric.**

This tutorial set walks the workflow end-to-end: from looking at the
weights, to running the AXI wrapper in cocotb, to using the deployed
overlay from Python over `pynq.MMIO`.

## What is special about this overlay

Most embedded transformer inference paths trade fabric for DRAM
streaming: weights live off-chip in DDR, and the FPGA runs one matmul
at a time. That gives flexibility at the cost of latency and host
orchestration complexity.

microgpt takes the opposite extreme: every one of the **4,192 INT16
(Q12) parameters** is hardcoded into LUTRAM / BRAM / constants at
**synthesis time**. There is no DDR, no DMA, no host-side inference
loop. The PS pushes a prompt token through a single AXI4-Lite slave,
the PL runs the full forward pass in fabric, and the PS reads the
generated tokens back through the same slave.

The exact-arithmetic GPT core is a port of
[`Luthiraa/TALOS-V2`](https://github.com/Luthiraa/TALOS-V2);
see [`UPSTREAM.md`](../UPSTREAM.md) for per-file attribution.

## The four-stage loop

```
    .hex weights   →   Vivado build   →   AXI4-Lite slave   →   PS driver
   (hw/ip/*.hex)      (hw/tcl/build       (microgpt_pynq_      (sw/drivers/
                       .tcl)               top.sv)              microgpt.py)
        │                  │                    │                    │
        ▼                  ▼                    ▼                    ▼
   visualise          synthesise          simulate &            generate
   (this notebook,    (Vivado 2024.1)     verify handshakes     tokens
    `01_…`)                                with cocotb           (notebook `03_…`
                                          (notebook `02_…`)      runs on PYNQ)
```

## Tutorials in this set

| # | Notebook | Runs locally? | Needs |
|---|---|---|---|
| 00 | This overview | n/a | nothing |
| 01 | `01_explore_weights.ipynb` | yes | `numpy`, `matplotlib` |
| 02 | `02_register_map_and_driver.ipynb` | yes (Python parts) | `numpy` |
| 03 | Cocotb simulation of the AXI wrapper | yes | `cocotb`, `verilator` (or `iverilog`) |

(03 is documented in [`hw/sim/cocotb/README.md`](../hw/sim/cocotb/README.md) and
run directly with `make` rather than from a notebook.)

## Reproducing the bitstream

```bash
source ~/tools/Xilinx/Vivado/2024.1/settings64.sh
rm -rf hw/build && mkdir hw/build
vivado -mode batch -source hw/tcl/build.tcl
```

Produces `overlays/microgpt.bit` and `.hwh` targeting
`xc7z010clg400-1`. Edit the `part` variable at the top of
`hw/tcl/build.tcl` to retarget for a `xc7z020` PYNQ-Z2 unit.

## Register map (4 KB AXI4-Lite BAR)

| Offset | Reg          | RW | Purpose                                          |
|--------|--------------|----|--------------------------------------------------|
| 0x000  | MAGIC        | RO | `'MGRT'` = 0x4D475254                            |
| 0x004  | VERSION      | RO | 0x00020001                                       |
| 0x008  | CMD          | WO | bit0=start pulse, bit1=clear pulse               |
| 0x00C  | STATUS       | RO | ready / busy / done / error / toggle / pos / out_len |
| 0x010  | CONFIG       | RW | temperature Q8.8 (hi 16b), max_gen (next 8b)      |
| 0x014  | SEED         | RW | RNG seed for the categorical sampler             |
| 0x018  | LOGIT_INFO   | RO | argmax token + last-sampled token + top-logit Q12 |
| 0x01C  | BOS          | RO | BOS_TOKEN (= 26)                                 |
| 0x060+ | OUTPUT_MEM   | RO | 16 generated tokens (low byte of each u32)        |
| 0x100+ | LOGITS       | RO | 27 sign-extended Q12 logits                       |

(Full register map and bitfield notes live in
[`README.md`](../README.md#register-map).)

## Going further

- The current build targets `xc7z010clg400-1` to fit the smaller
  PYNQ-Z2 die. The model fits comfortably; a larger model could
  potentially run on the `xc7z020` die without restructuring.
- `sw/notebooks/throughput.ipynb` measures per-token latency on the
  deployed board and shows the ~1.7× speedup from the burst-readback
  + UIO IRQ driver optimisations.
- The categorical sampler uses `xorshift32` for reproducibility:
  `gpt.generate(seed=0xC0FFEE)` returns deterministic text.
